# Does NCA pre-pre-training accelerate subsequent language learning?

**Version 2: independent stage budgets.** First validate synthetic prediction on
unseen NCA rules, then compare the same language-training procedure from random,
grid-NCA and shuffled-NCA initializations. A positive result is not guaranteed.
The graph extension is deferred until the grid reference works.

Before Run All: select one available GPU accelerator and enable Internet.
The notebook uses one GPU, downloads pinned WikiText train/validation data and
GPT-2 tokenizer assets, and leaves the official confirmation test split untouched.
Default budgets are 1000 PPT updates and **2000 equal language updates** per arm.
Three paired seeds. A small RoPE/RMSNorm/SwiGLU transformer, categorical 10-state
NCA, complexity-filtered rules and ten burn-in steps replace pilot 1's toy setup.

If held-out synthetic learning fails the declared gate, language training is
skipped and diagnostic outputs are still exported. Passing the gate is only
permission to test transfer, not evidence that it works. The final ZIP contains
checkpoints, source, configuration, rule/data identities, logs and verification.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timezone

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
ROOT = Path('/kaggle/working') / f'nca_transfer_v2_{RUN_ID}'
ROOT.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
os.environ['TIKTOKEN_CACHE_DIR'] = str(ROOT / 'tokenizer_cache')
os.environ['MPLCONFIGDIR'] = str(ROOT / 'plot_cache')
print('Run:', RUN_ID, 'Workspace:', ROOT)

for directory in ['configs', 'docs', 'sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431', 'tests', 'transfer_v2']:
    Path(directory).mkdir(parents=True, exist_ok=True)


## Embedded, reviewable sources
These files match the locally checked v2 source. No Git checkout is required.

In [ ]:
%%writefile transfer_v2/__init__.py
"""Independent pre-pre-training and language-stage transfer experiment (v2)."""


In [ ]:
%%writefile transfer_v2/calibrate.py
"""Disposable on-device timing calibration; never changes a research model."""
import hashlib
from pathlib import Path
import statistics
import time

import numpy as np
import torch

from .common import ARMS, dump, sync
from .model import LM, language_model
from .training import evaluate, optimizer_for, update


def calibrate(c, arrays, manifest, device, output):
    values = {}
    for name, synthetic in (("ppt", True), ("language", False)):
        stage = c[name]
        model = (LM(c["model"], c["nca"]["states"]**4+2, 0, 1) if synthetic else
                 language_model(c, manifest["text"]["vocab"], 0)).to(device)
        optimizer = optimizer_for(model, stage, synthetic)
        array = arrays["nca_train"] if synthetic else arrays["text_train"]
        validation = arrays["nca_validation"] if synthetic else arrays["text_validation"]
        rng = np.random.default_rng(0)
        order_hash = hashlib.sha256()
        measurements = []
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        for step in range(4):
            sync(device)
            started = time.perf_counter()
            update(model, optimizer, array, c, stage, min(step, stage["steps"]-1), rng, device, synthetic, order_hash)
            sync(device)
            if step:
                measurements.append(time.perf_counter()-started)
        started = time.perf_counter()
        evaluate(model, validation, c, device, synthetic)
        sync(device)
        values[name] = {"seconds_per_update_median": statistics.median(measurements),
                        "validation_seconds": time.perf_counter()-started,
                        "parameters": sum(p.numel() for p in model.parameters()),
                        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated()/1e6 if device == "cuda" else None}
        del model, optimizer
    count = len(c["seeds"])
    approx = 0.
    for name, jobs in (("ppt", 2*count), ("language", len(ARMS)*count)):
        steps = c[name]["steps"]
        evals = (steps+c["eval_every"]-1)//c["eval_every"]+1
        approx += jobs*(steps*values[name]["seconds_per_update_median"]+evals*values[name]["validation_seconds"])
    result = {"purpose": "Disposable warmup/timing calibration only; no checkpoint used for transfer",
              "device": device, "stage_measurements": values,
              "approx_full_training_and_validation_minutes": approx/60,
              "excludes": "Generation, checkpoint saves, verification, plotting, downloads, queueing and changing GPU load; not a runtime guarantee"}
    dump(Path(output)/"calibration.json", result)
    print(f"Measured estimate for full training + validation: {approx/60:.1f} minutes, excluding preparation/checkpoint/verification/archive work.", flush=True)
    return result


In [ ]:
%%writefile transfer_v2/common.py
from __future__ import annotations

import hashlib
import json
import math
from pathlib import Path
import random

import numpy as np
import torch

ARMS = ("scratch", "grid_nca", "shuffled_nca")
REFERENCE_COMMIT = "bdd1c71e04eb2e5a2c91481faf278e95846b4431"


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False)+"\n", encoding="utf-8")
    temporary.replace(path)


def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


def state_sha(state):
    h = hashlib.sha256()
    for name, tensor in sorted(state.items()):
        h.update(name.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def source_hashes():
    root = Path(__file__).resolve().parent
    return {p.name: sha_file(p) for p in sorted(root.glob("*.py"))}


def sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_config(path):
    c = json.loads(Path(path).read_text(encoding="utf-8"))
    validate_config(c)
    return c


def validate_config(c):
    if c["schema_version"] != 2:
        raise ValueError("Expected an independent-stage v2 configuration")
    if c["purpose"] not in ("development", "execution_check"):
        raise ValueError("Unknown run purpose")
    if not c["seeds"] or len(set(c["seeds"])) != len(c["seeds"]):
        raise ValueError("Model seeds must be unique")
    m, n = c["model"], c["nca"]
    if m["width"] % m["heads"] or (m["width"]//m["heads"]) % 2:
        raise ValueError("RoPE requires an even head dimension")
    if n["grid"] % 2 or n["states"] < 2:
        raise ValueError("Expected an even grid and a categorical alphabet")
    if n["states"]**4+2 > 65536:
        raise ValueError("Synthetic token IDs must fit uint16")
    if m["context"] < 2*frame_length(c):
        raise ValueError("Context must contain at least two complete frames")
    if any(c[stage]["steps"] <= 0 for stage in ("ppt", "language")):
        raise ValueError("PPT and language budgets are separate positive lengths")
    if not 0 <= n["filter_low"] < n["filter_high"]:
        raise ValueError("Invalid complexity band")
    if c["purpose"] == "development" and not c["gate"]["enforce"]:
        raise ValueError("Development runs must enforce the held-out synthetic gate")
    if c["purpose"] == "development" and c["text"]["tokenizer"] != "gpt2":
        raise ValueError("Byte fixtures are restricted to execution checks")
    if c["precision"] != "fp32":
        raise ValueError("This audited version uses fp32 in both stages")


def frame_length(c):
    return (c["nca"]["grid"]//2)**2+2


def effective_tokens(c):
    return c["batch"]*c["accumulate"]*c["model"]["context"]


def lr_at(step, stage):
    length = stage["steps"]
    warm = max(1, min(length, round(length*stage["warmup_fraction"])))
    if step < warm:
        return stage["lr"]*(step+1)/warm
    fraction = (step-warm)/max(1, length-warm-1)
    return stage["lr"]*(.1+.9*(1+math.cos(math.pi*fraction))/2)


In [ ]:
%%writefile transfer_v2/data.py
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time
import urllib.request

import numpy as np

from .common import dump, sha_file, source_hashes
from .nca import generate_split, shuffled


def download(url, path):
    temporary = Path(str(path)+".part")
    req = urllib.request.Request(url, headers={"User-Agent": "NCA-prepretraining-transfer-v2"})
    try:
        with urllib.request.urlopen(req, timeout=90) as src, temporary.open("wb") as dst:
            while chunk := src.read(1024*1024):
                dst.write(chunk)
        temporary.replace(path)
    except Exception:
        # Preserve failed download for diagnosis; never consume a partial file.
        raise


def prepare_text(c, root, fixture_dir=None):
    started = time.perf_counter()
    provenance = {"source": c["text"], "splits": {}, "tokenizer": c["text"]["tokenizer"]}
    if fixture_dir is not None:
        if c["purpose"] != "execution_check":
            raise ValueError("Synthetic local text fixtures are not research data")
        splits = ["train", "validation"]+(["test"] if c["text"]["evaluate_test"] else [])
        for split in splits:
            raw = (Path(fixture_dir)/f"{split}.txt").read_bytes()
            np.save(root / f"text_{split}.npy", np.frombuffer(raw, dtype=np.uint8).astype(np.uint16))
            provenance["splits"][split] = {"bytes_sha256": hashlib.sha256(raw).hexdigest(), "tokens": len(raw)}
        provenance["vocab"] = 256
        provenance["test_status"] = "execution_fixture_only" if "test" in splits else "not_loaded"
    else:
        if c["text"]["tokenizer"] != "gpt2":
            raise ValueError("Execution checks require an explicit --fixture-dir")
        import pyarrow.parquet as pq
        os.environ.setdefault("TIKTOKEN_CACHE_DIR", str((root/"tokenizer_cache").resolve()))
        import tiktoken
        encoder = tiktoken.get_encoding("gpt2")
        provenance["vocab"] = encoder.n_vocab
        provenance["tokenizer_package_version"] = importlib.metadata.version("tiktoken")
        provenance["parquet_reader_version"] = importlib.metadata.version("pyarrow")
        # Fingerprint the actual merge table + special tokens, not just its name.
        merge = sorted((token.hex(), rank) for token, rank in encoder._mergeable_ranks.items())
        provenance["tokenizer_sha256"] = hashlib.sha256(json.dumps([merge, encoder._special_tokens], separators=(",", ":")).encode()).hexdigest()
        splits = ["train", "validation"]+(["test"] if c["text"]["evaluate_test"] else [])
        for split in splits:
            url = (f"https://huggingface.co/datasets/{c['text']['dataset']}/resolve/{c['text']['revision']}/"
                   f"{c['text']['subset']}/{split}-00000-of-00001.parquet")
            path = root / f"wikitext_{split}.parquet"
            download(url, path)
            rows = pq.read_table(path, columns=["text"])["text"].to_pylist()
            raw = "\n".join(rows)
            ids = np.array(encoder.encode_ordinary(raw), dtype=np.uint16)
            np.save(root / f"text_{split}.npy", ids)
            provenance["splits"][split] = {"url": url, "parquet_sha256": sha_file(path),
                                           "text_sha256": hashlib.sha256(raw.encode()).hexdigest(),
                                           "tokens": len(ids), "rows": len(rows)}
        provenance["test_status"] = "loaded_for_predeclared_final_evaluation" if "test" in splits else "reserved_not_downloaded"
        provenance["license"] = "WikiText: CC-BY-SA-3.0 / GFDL; see Salesforce/wikitext dataset card"
    provenance["seconds"] = time.perf_counter()-started
    return provenance


def prepare(c, root, device, fixture_dir=None):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    import torch
    manifest = {"config": c, "text": prepare_text(c, root, fixture_dir), "nca": {}, "files": {},
                "generation": {"device": str(device), "torch": str(torch.__version__), "numpy": np.__version__,
                               "source_sha256": source_hashes()}}
    for split, count in (("train", c["nca"]["train_rules"]), ("validation", c["nca"]["val_rules"])):
        tick_prepare = time.perf_counter()
        try:
            tokens, rules, stats = generate_split(c, count, split, device)
        except Exception as exc:
            dump(root / "PREPARATION_FAILED.json", {"stage": split, "error": str(exc), "config": c})
            raise
        np.save(root / f"nca_{split}.npy", tokens)
        np.savez(root / f"rules_{split}.npz", **rules)
        tick = time.perf_counter()
        np.save(root / f"shuffled_{split}.npy", shuffled(tokens, c, c["nca"]["data_seed"]+(10 if split == "train" else 20)))
        stats["shuffle_seconds"] = time.perf_counter()-tick
        stats["prepare_seconds"] = time.perf_counter()-tick_prepare
        manifest["nca"][split] = stats
    train_ids = set(manifest["nca"]["train"]["rule_ids"])
    val_ids = set(manifest["nca"]["validation"]["rule_ids"])
    if train_ids & val_ids:
        raise RuntimeError("Train/validation rule identities overlap")
    for path in sorted(root.iterdir()):
        if path.suffix in (".npy", ".npz", ".parquet"):
            manifest["files"][path.name] = sha_file(path)
    manifest["total_prepare_seconds"] = time.perf_counter()-started
    manifest["shared_finalize_seconds"] = max(0., manifest["total_prepare_seconds"]-manifest["text"]["seconds"]
                                               -sum(s["prepare_seconds"] for s in manifest["nca"].values()))
    dump(root / "manifest.json", manifest)
    return manifest


def load_data(root, c):
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    if manifest["config"] != c:
        raise ValueError("Cached data/config mismatch")
    for name, expected in manifest["files"].items():
        if sha_file(root / name) != expected:
            raise ValueError(f"Cached file checksum mismatch: {name}")
    arrays = {path.stem: np.load(path, allow_pickle=False) for path in root.glob("*.npy")}
    return arrays, manifest


In [ ]:
%%writefile transfer_v2/model.py
"""Small RoPE/RMSNorm/SwiGLU causal transformer; no pretrained external weights."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

from .common import state_sha


class RMSNorm(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))

    def forward(self, x):
        return x*torch.rsqrt(x.float().square().mean(-1, keepdim=True)+1e-6).to(x.dtype)*self.weight


def rope(x):
    # Interleaved real/imaginary coordinates; equivalent rotary formulation.
    t, d = x.shape[-2:]
    frequency = 10000.0**(-torch.arange(0, d, 2, device=x.device).float()/d)
    angle = torch.arange(t, device=x.device).float()[:, None]*frequency[None]
    a, b = x[..., 0::2], x[..., 1::2]
    co, si = angle.cos().to(x.dtype), angle.sin().to(x.dtype)
    return torch.stack((a*co-b*si, a*si+b*co), -1).flatten(-2)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        d, h = config["width"], config["heads"]
        hidden = ((int(8*d/3)+63)//64)*64
        self.heads = h
        self.attention_norm, self.ffn_norm = RMSNorm(d), RMSNorm(d)
        self.qkv = nn.Linear(d, 3*d, bias=False)
        self.output = nn.Linear(d, d, bias=False)
        self.gate, self.up, self.down = (nn.Linear(d, hidden, bias=False),
                                       nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False))

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.attention_norm(x)).reshape(b, t, 3, self.heads, d//self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(rope(q), rope(k), v, dropout_p=0., is_causal=True)
        x = x+self.output(a.transpose(1, 2).reshape(b, t, d))
        z = self.ffn_norm(x)
        return x+self.down(F.silu(self.gate(z))*self.up(z))


class Backbone(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config["layers"])])
        self.norm = RMSNorm(config["width"])

    def forward(self, x):
        return self.norm(self.blocks(x))


class LM(nn.Module):
    def __init__(self, config, vocab, core_seed, head_seed):
        super().__init__()
        # Separate streams ensure vocabulary-size changes cannot change the
        # paired initial backbone. Construction happens on CPU, then moves.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(core_seed)
            self.core = Backbone(config)
            for module in self.core.modules():
                if isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, std=.02)
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(head_seed)
            self.embedding = nn.Embedding(vocab, config["width"])
            nn.init.normal_(self.embedding.weight, std=.02)
        self.vocab = vocab
        self.context = config["context"]

    def forward(self, tokens):
        if tokens.shape[1] > self.context:
            raise ValueError("Sequence exceeds declared context")
        z = self.core(self.embedding(tokens))
        return F.linear(z, self.embedding.weight)  # Explicitly tied input/output table.

    def core_hash(self):
        return state_sha(self.core.state_dict())

    def head_hash(self):
        return state_sha(self.embedding.state_dict())


def language_model(c, vocab, seed, ppt_state=None):
    model = LM(c["model"], vocab, core_seed=seed, head_seed=seed+100_000)
    if ppt_state is not None:
        # Never transfer synthetic embeddings, optimizer moments, or scheduler.
        model.core.load_state_dict(ppt_state, strict=True)
    return model


In [ ]:
%%writefile transfer_v2/nca.py
"""Categorical spatial NCA port, frozen random rules and rule-disjoint datasets.

Reference: danihyunlee/nca-pre-pretraining at bdd1c71e04eb2e5a2c91481faf278e95846b4431.
This implements spatial-only padding and uniform initial states as described in
the paper, rather than inheriting the reference's channel-padding side effect.
"""
from __future__ import annotations

import gzip
import math
import time

import numpy as np
import torch
from torch.nn import functional as F

from .common import frame_length, sync


def lattice(side):
    ids = np.arange(side*side).reshape(side, side)
    return np.stack([np.roll(ids, (-dy, -dx), (0, 1)).ravel()
                     for dy in (-1, 0, 1) for dx in (-1, 0, 1)], 1)


def sample_rules(count, states, generator, device):
    def lecun(shape, fan_in):
        # Flax-style fan-in variance with a normal truncated to [-2,2].
        u = torch.rand(shape, generator=generator, device=device)*.9544997361+.02275013195
        return (2*u-1).erfinv()*math.sqrt(2/fan_in)/.8796256610
    return {"perception": lecun((count, 9*states, 4), 9*states),
            "hidden": lecun((count, 4, 16), 4),
            "output": lecun((count, 16, states), 16)}


def message_logits(state, edges, rules, states):
    local = F.one_hot(state[:, edges], states).float().flatten(-2)
    perception = torch.einsum("bni,bic->bnc", local, rules["perception"])
    hidden = F.relu(torch.einsum("bnc,bch->bnh", perception, rules["hidden"]))
    return torch.einsum("bnh,bhs->bns", hidden, rules["output"])


def convolution_logits(state, rules, side, states):
    # Independent spatial-convolution formulation for the equivalence test.
    b = state.shape[0]
    x = F.one_hot(state, states).float().reshape(b, side, side, states).permute(0, 3, 1, 2)
    weight = rules["perception"].reshape(b, 9, states, 4).permute(0, 3, 2, 1).reshape(b*4, states, 3, 3)
    perceived = F.conv2d(F.pad(x.reshape(1, b*states, side, side), (1, 1, 1, 1), mode="circular"), weight, groups=b)
    p = perceived.reshape(b, 4, side*side).transpose(1, 2)
    hidden = F.relu(torch.einsum("bnc,bch->bnh", p, rules["hidden"]))
    return torch.einsum("bnh,bhs->bns", hidden, rules["output"])


def step(state, edges, rules, states, temperature, generator):
    logits = message_logits(state, edges, rules, states)
    if temperature > 0:
        u = torch.rand(logits.shape, device=logits.device, generator=generator).clamp(1e-7, 1-1e-7)
        logits = logits-temperature*torch.log(-torch.log(u))
    return logits.argmax(-1)


def rollout(rules, c, frames, generator, device):
    n = c["nca"]
    edges = torch.as_tensor(lattice(n["grid"]), dtype=torch.long, device=device)
    batch = rules["perception"].shape[0]
    state = torch.randint(n["states"], (batch, n["grid"]**2), generator=generator, device=device)
    for _ in range(n["burnin"]):
        state = step(state, edges, rules, n["states"], n["temperature"], generator)
    values = []
    for _ in range(frames):
        values.append(state)
        state = step(state, edges, rules, n["states"], n["temperature"], generator)
    return torch.stack(values, 1)


def encode_frames(frames, side, states):
    x = frames.reshape(*frames.shape[:-1], side, side)
    patches = (x[..., 0::2, 0::2]+states*x[..., 0::2, 1::2]
               +states**2*x[..., 1::2, 0::2]+states**3*x[..., 1::2, 1::2]).flatten(-2)
    start = torch.full((*patches.shape[:-1], 1), states**4, dtype=torch.long, device=frames.device)
    return torch.cat((start, patches, start+1), -1).flatten(1)


def complexity(tokens, c):
    patches = tokens.reshape(tokens.shape[0], -1, frame_length(c))[:, :, 1:-1]
    scores = []
    for row in patches:
        raw = np.asarray(row, dtype="<i4").tobytes(order="C")
        scores.append(len(gzip.compress(raw, compresslevel=9, mtime=0))/len(raw))
    return np.array(scores)


@torch.no_grad()
def generate_split(c, count, split, device):
    n = c["nca"]
    if n["channel_padding"] != "spatial_only":
        raise ValueError("This port explicitly uses spatial-only periodic padding")
    seed = n["data_seed"]+(0 if split == "train" else 1_000_003)
    generator = torch.Generator(device=device).manual_seed(seed)
    frames = math.ceil((c["model"]["context"]+1)/frame_length(c))
    arrays, saved_rules, ids, profile_scores, rollout_scores, fixed = [], {}, [], [], [], []
    candidates = accepted = band_passes = 0
    sync(device)
    started = time.perf_counter()
    while accepted < count:
        b = min(n["candidate_batch"], n["max_candidates"]-candidates)
        if b <= 0:
            raise RuntimeError(f"{split}: complexity filter exhausted {candidates} candidates after accepting {accepted}/{count}; no silent threshold relaxation")
        rules = sample_rules(b, n["states"], generator, device)
        profile = rollout(rules, c, n["profile_frames"], generator, device)
        profile_tokens = encode_frames(profile, n["grid"], n["states"]).cpu().numpy()
        scores = complexity(profile_tokens, c)
        eligible = np.flatnonzero((scores > n["filter_low"]) & (scores < n["filter_high"]))
        band_passes += len(eligible)
        selected = eligible[:count-accepted]
        candidate_ids = np.arange(candidates, candidates+b)
        candidates += b
        if len(selected):
            chosen = {k: v[selected.tolist()] for k, v in rules.items()}
            # Fresh initial conditions for training/validation trajectories;
            # screening is on an independent rollout of each selected rule.
            actual = rollout(chosen, c, frames, generator, device)
            actual_tokens = encode_frames(actual, n["grid"], n["states"]).cpu().numpy().astype(np.uint16)
            arrays.append(actual_tokens)
            for key, value in chosen.items():
                saved_rules.setdefault(key, []).append(value.cpu().numpy())
            ids.extend([f"{split}:{seed}:{int(i)}" for i in candidate_ids[selected]])
            profile_scores.extend(scores[selected].tolist())
            rollout_scores.extend(complexity(actual_tokens, c).tolist())
            fixed.extend((actual[:, -1] == actual[:, -2]).all(-1).cpu().tolist())
            accepted += len(selected)
        if candidates % (n["candidate_batch"]*20) == 0 or accepted == count:
            print(f"NCA {split}: accepted {accepted}/{count}, screened {candidates} rules", flush=True)
    sync(device)
    stats = {"screening_seed": seed, "rule_ids": ids, "candidates": candidates, "accepted": accepted,
             "retained_per_candidate": accepted/candidates, "band_passes": band_passes,
             "band_pass_rate": band_passes/candidates, "seconds": time.perf_counter()-started,
             "profile_gzip_quartiles": np.quantile(profile_scores, [.25, .5, .75]).tolist(),
             "fresh_rollout_gzip_quartiles": np.quantile(rollout_scores, [.25, .5, .75]).tolist(),
             "final_fixed_transition_fraction": float(np.mean(fixed)),
             "profile_scores": profile_scores,
             "serialization": "Per-rule marker-free little-endian int32 patch IDs; gzip level 9, mtime 0",
             "filter_note": "Rules selected on independent profile initial conditions; fresh rollout scores need not remain in the screening band."}
    return np.concatenate(arrays), {k: np.concatenate(v) for k, v in saved_rules.items()}, stats


def shuffled(tokens, c, seed):
    result = tokens.copy()
    rng = np.random.default_rng(seed)
    cutoff = c["nca"]["states"]**4
    for row in result:
        mask = row < cutoff
        row[mask] = rng.permutation(row[mask])
    return result


def targets(tokens, c, patch_only=False):
    labels = tokens[:, 1:].clone()
    absolute = torch.arange(1, tokens.shape[1], device=tokens.device)
    mask = absolute < frame_length(c)
    labels[:, mask] = -100
    if patch_only:
        labels[labels >= c["nca"]["states"]**4] = -100
    return labels


def control_probabilities(train, c):
    cutoff, length = c["nca"]["states"]**4, c["model"]["context"]
    # Fit controls on the SAME trained prefix, never on held-out rules.
    prefix = train[:, :length+1].astype(np.int64)
    y = prefix[:, 1:]
    absolute = np.arange(1, length+1)
    mask = (absolute[None] >= frame_length(c)) & (y < cutoff)
    counts = np.bincount(y[mask], minlength=cutoff).astype(np.float64)+1
    probabilities = counts/counts.sum()
    previous = prefix[:, np.maximum(0, absolute-frame_length(c))]
    truth, before = y[mask], previous[mask]
    base = probabilities[truth]
    copy = (truth == before).astype(np.float64)
    mixture_grid = np.linspace(0, .99, 100)
    losses = [-np.log((1-q)*base+q*copy).mean() for q in mixture_grid]
    mixture = float(mixture_grid[int(np.argmin(losses))])
    return probabilities, mixture


def evaluate_controls(tokens, c, probabilities, mixture):
    length, f = c["model"]["context"], frame_length(c)
    prefix = tokens[:, :length+1].astype(np.int64)
    y = prefix[:, 1:]
    absolute = np.arange(1, length+1)
    mask = (absolute[None] >= f) & (y < len(probabilities))
    previous = prefix[:, np.maximum(0, absolute-f)]
    truth, before = y[mask], previous[mask]
    base = probabilities[truth]
    mixture_probability = (1-mixture)*base+mixture*(truth == before)
    return {"unigram_patch_nll": float(-np.log(base).mean()),
            "copy_mixture_patch_nll": float(-np.log(mixture_probability).mean()),
            "copy_mixture_weight_fitted_on_train": mixture,
            "unigram_patch_accuracy": float(np.mean(truth == probabilities.argmax())),
            "previous_frame_patch_accuracy": float(np.mean(truth == before)),
            "patch_targets": int(mask.sum())}


In [ ]:
%%writefile transfer_v2/report.py
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
import statistics

import numpy as np

from .common import ARMS, dump


def first_hit(curve, threshold):
    return next((r for r in curve if r["validation_nll"] <= threshold), None)


def report(root, c):
    root = Path(root)
    status = json.loads((root/"status.json").read_text())
    summaries, curves = {}, {}
    for arm in ARMS:
        for seed in c["seeds"]:
            folder = root/f"{arm}_{seed}"
            if (folder/"language_summary.json").exists():
                summaries[arm, seed] = json.loads((folder/"language_summary.json").read_text())
                curves[arm, seed] = [json.loads(s) for s in (folder/"language_metrics.jsonl").read_text().splitlines()]
    comparisons = []
    for seed in c["seeds"]:
        if ("scratch", seed) not in summaries:
            continue
        reference = summaries["scratch", seed]["final_validation_nll"]
        base_hit = first_hit(curves["scratch", seed], reference)
        for arm in ("grid_nca", "shuffled_nca"):
            if (arm, seed) not in summaries:
                continue
            hit = first_hit(curves[arm, seed], reference)
            row = {"arm": arm, "seed": seed, "reference_nll": reference,
                   "reference_policy": "Paired scratch final validation loss at its declared N language updates",
                   "reference_reached": hit is not None,
                   "language_step_at_reference": hit["language_step"] if hit else None,
                   "final_validation_nll_delta": summaries[arm, seed]["final_validation_nll"]-reference}
            for key in ("language_tokens", "language_train_seconds", "language_seconds", "total_training_tokens", "cumulative_seconds"):
                row[f"{key}_speedup"] = base_hit[key]/hit[key] if hit and hit[key] > 0 else None
            comparisons.append(row)
    aggregate = []
    for arm in ("grid_nca", "shuffled_nca"):
        rows = [r for r in comparisons if r["arm"] == arm]
        if not rows:
            continue
        deltas = [r["final_validation_nll_delta"] for r in rows]
        a = {"arm": arm, "paired_seeds": len(rows), "seeds_reaching_reference": sum(r["reference_reached"] for r in rows),
             "mean_final_validation_nll_delta": statistics.mean(deltas)}
        for key in ("language_tokens", "language_seconds", "cumulative_seconds"):
            values = [r[f"{key}_speedup"] for r in rows]
            # Do not publish a success-only mean when any seed is censored.
            complete = all(v is not None and v > 0 for v in values)
            a[f"geomean_{key}_speedup_all_seeds"] = math.exp(statistics.mean(math.log(v) for v in values)) if complete else None
            if complete and len(values) >= 2:
                rng = np.random.default_rng(101)
                samples = np.exp(np.mean(rng.choice(np.log(values), (10000, len(values)), replace=True), axis=1))
                a[f"exploratory_95_ci_{key}"] = np.quantile(samples, [.025, .975]).tolist()
            else:
                a[f"exploratory_95_ci_{key}"] = None
        aggregate.append(a)
    result = {"schema_version": 2, "purpose": c["purpose"], "status": status,
              "primary_question": "Does NCA pre-pre-training accelerate learning in an identical language stage?",
              "secondary_question": "Do language-stage savings repay PPT and generation costs?",
              "graph_status": "Not run: establish grid transfer before adding the graph treatment",
              "comparisons": comparisons, "aggregate": aggregate,
              "uncertainty_note": "Three model seeds on one shared synthetic dataset are exploratory. Evaluation crossings are discrete, without interpolation. A synthetic gate pass is not proof of transfer.",
              "test_policy": "Reserved and not loaded" if not c["text"]["evaluate_test"] else "Predeclared final evaluation only"}
    dump(root/"report.json", result)
    plot_ppt(root, c)
    if summaries:
        flat = [{k: v for k, v in s.items() if not isinstance(v, dict)} for s in summaries.values()]
        with (root/"summary.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(flat[0]))
            w.writeheader()
            w.writerows(flat)
        plot(root, c, curves)
    print(json.dumps({"status": status, "aggregate": aggregate}, indent=2), flush=True)
    return result


def plot_ppt(root, c):
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for arm, color in (("grid_nca", "#2563eb"), ("shuffled_nca", "#db2777")):
        for i, seed in enumerate(c["seeds"]):
            folder = Path(root)/f"{arm}_{seed}"
            if not (folder/"ppt_summary.json").exists():
                continue
            rows = [json.loads(s) for s in (folder/"ppt_metrics.jsonl").read_text().splitlines()]
            summary = json.loads((folder/"ppt_summary.json").read_text())
            label = arm if i == 0 else None
            trained = [r for r in rows if r["train_loss"] is not None]
            axes[0].plot([r["ppt_step"] for r in trained], [r["train_loss"] for r in trained], color=color, alpha=.65, label=label)
            axes[1].plot([r["ppt_step"] for r in rows], [r["validation"]["patch_nll"] for r in rows], color=color, alpha=.65)
            axes[2].plot([r["ppt_step"] for r in rows], [r["validation"]["patch_accuracy"] for r in rows], color=color, alpha=.65)
            if i == 0:
                bound = min(summary["controls"]["unigram_patch_nll"], summary["controls"]["copy_mixture_patch_nll"])
                axes[1].axhline(bound, color=color, linestyle="--", linewidth=.8)
    for ax, title, ylabel in zip(axes, ("Training objective", "Unseen-rule patch prediction", "Unseen-rule accuracy"),
                                 ("Cross-entropy (conditioning masked)", "Patch NLL (dashed: best fitted control)", "Patch accuracy")):
        ax.set_title(title)
        ax.set_xlabel("PPT updates")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    fig.suptitle("Synthetic-stage diagnostics — a gate pass does not establish language transfer")
    fig.savefig(Path(root)/"ppt_diagnostics.png", dpi=180)
    fig.savefig(Path(root)/"ppt_diagnostics.svg")
    plt.close(fig)


def plot(root, c, curves):
    import matplotlib
    matplotlib.use("Agg")
    from matplotlib import pyplot as plt
    colors = {"scratch": "#64748b", "grid_nca": "#2563eb", "shuffled_nca": "#db2777"}
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    keys = ("language_tokens", "language_seconds", "cumulative_seconds")
    for (arm, seed), rows in curves.items():
        for ax, key in zip(axes, keys):
            scale = 1e6 if key == "language_tokens" else 1
            ax.plot([r[key]/scale for r in rows], [r["validation_nll"] for r in rows],
                    color=colors[arm], alpha=.6, label=arm if seed == c["seeds"][0] else None)
    for ax, title, xlabel in zip(axes, ("Language exposure", "Language-stage time", "PPT + language cost"),
                                 ("Language tokens (millions)", "Seconds since language step 0", "Recorded cumulative seconds")):
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Validation NLL (nats per language token)")
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    prefix = "Execution fixture; no research inference" if c["purpose"] == "execution_check" else "Independent-stage NCA pre-pre-training transfer"
    fig.suptitle(prefix)
    fig.savefig(root/"learning_curves.png", dpi=180)
    fig.savefig(root/"learning_curves.svg")
    plt.close(fig)


In [ ]:
%%writefile transfer_v2/run.py
"""CLI: python -m transfer_v2.run all --config configs/transfer_v2.json ..."""
from __future__ import annotations

import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import argparse
from datetime import datetime, timezone
import json
from pathlib import Path
import platform
import sys
import time

import numpy as np
import torch

from .common import ARMS, REFERENCE_COMMIT, dump, load_config, sha_file, source_hashes, sync
from .calibrate import calibrate
from .data import load_data, prepare
from .report import report
from .training import train_language, train_ppt
from .verify import verify


def setup(device):
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but no CUDA GPU is available")


def initialize_run(c, root, data_root, device):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=False)
    dump(root/"environment.json", {
        "schema_version": 2, "config": c, "started_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version, "torch": str(torch.__version__), "platform": platform.platform(),
        "numpy": np.__version__,
        "device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "cuda": torch.version.cuda, "tf32": False, "reference_commit": REFERENCE_COMMIT,
        "source_sha256": source_hashes(), "data_manifest_sha256": sha_file(Path(data_root)/"manifest.json"),
        "graph_treatment": "Deferred until grid transfer is established",
        "timing_scope": "Language seconds start after model/head/optimizer construction and include validation. Cumulative adds text preparation, relevant synthetic preparation/cache writes, head/optimizer construction, shared manifest work and measured PPT stage. Excludes package startup, cloud queue, optional disposable calibration and final language checkpoint/verification/plot/archive work."})
    dump(root/"status.json", {"state": "ppt_pending", "purpose": c["purpose"]})


def check_identity(root, c, data_root):
    env = json.loads((Path(root)/"environment.json").read_text())
    if env["config"] != c or env["data_manifest_sha256"] != sha_file(Path(data_root)/"manifest.json"):
        raise ValueError("Run/config/data mismatch")
    if env["source_sha256"] != source_hashes():
        raise ValueError("Source changed during this staged run; create a fresh run")


def ppt_stage(c, root, arrays, manifest_hash, device):
    summaries = []
    dump(Path(root)/"status.json", {"state": "ppt_running", "purpose": c["purpose"]})
    for i, seed in enumerate(c["seeds"]):
        order = ("grid_nca", "shuffled_nca") if i % 2 == 0 else ("shuffled_nca", "grid_nca")
        for arm in order:
            summaries.append(train_ppt(c, arm, seed, arrays, manifest_hash, root, device))
    failed = [s["seed"] for s in summaries if s["arm"] == "grid_nca" and not s["gate"]["passed"]]
    allowed = not failed or (c["purpose"] == "execution_check" and not c["gate"]["enforce"])
    status = {"state": "ppt_ready" if allowed else "stopped_at_synthetic_gate", "purpose": c["purpose"],
              "grid_seeds_failing_gate": failed, "language_allowed": allowed,
              "execution_only_bypass": bool(failed and allowed),
              "note": "Passing this gate only permits testing transfer; it does not establish acceleration."}
    dump(Path(root)/"status.json", status)
    dump(Path(root)/"ppt_gate.json", {"status": status, "summaries": summaries})
    return allowed


def language_stage(c, root, arrays, manifest, manifest_hash, device):
    status = json.loads((Path(root)/"status.json").read_text())
    if status["state"] != "ppt_ready" or not status["language_allowed"]:
        raise ValueError("Held-out synthetic diagnostics must permit language training first")
    dump(Path(root)/"status.json", {"state": "language_running", "purpose": c["purpose"]})
    for i, seed in enumerate(c["seeds"]):
        order = ARMS[i % len(ARMS):]+ARMS[:i % len(ARMS)]
        summaries = []
        for arm in order:
            summaries.append(train_language(c, arm, seed, arrays, manifest, manifest_hash, root, device))
        for key in ("core_initial_sha256", "language_head_initial_sha256", "language_train_order_sha256", "language_schedule_sha256"):
            if len({s[key] for s in summaries}) != 1:
                raise RuntimeError(f"Language controls failed: {key}")
    dump(Path(root)/"status.json", {"state": "complete", "purpose": c["purpose"],
                                  "scientific_claim": "Inspect paired language-stage curves and censored reference crossings; completion is not success."})


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("command", choices=("prepare", "calibrate", "ppt", "language", "report", "verify", "all"))
    p.add_argument("--config", default="configs/transfer_v2.json")
    p.add_argument("--data", default="data/transfer_v2")
    p.add_argument("--out", default="runs/transfer_v2")
    p.add_argument("--device", choices=("cpu", "cuda", "auto"), default="auto")
    p.add_argument("--fixture-dir", help="Execution checks only: train.txt and validation.txt")
    a = p.parse_args()
    c = load_config(a.config)
    device = "cuda" if a.device == "auto" and torch.cuda.is_available() else "cpu" if a.device == "auto" else a.device
    setup(device)
    if a.command in ("prepare", "all"):
        prepare(c, a.data, device, a.fixture_dir)
    if a.command == "all":
        arrays, manifest = load_data(a.data, c)
        calibrate(c, arrays, manifest, device, a.data)
    if a.command == "calibrate":
        arrays, manifest = load_data(a.data, c)
        calibrate(c, arrays, manifest, device, a.data)
        return
    if a.command in ("ppt", "all"):
        arrays, manifest = load_data(a.data, c)
        initialize_run(c, a.out, a.data, device)
        allowed = ppt_stage(c, a.out, arrays, sha_file(Path(a.data)/"manifest.json"), device)
        if not allowed:
            report(a.out, c)
            verify(a.out, a.data, device)
            print("STOPPED AT SYNTHETIC GATE: diagnostics saved; language benchmark not launched.", flush=True)
            return
    if a.command in ("language", "all"):
        check_identity(a.out, c, a.data)
        arrays, manifest = load_data(a.data, c)
        language_stage(c, a.out, arrays, manifest, sha_file(Path(a.data)/"manifest.json"), device)
    if a.command in ("language", "report", "all"):
        report(a.out, c)
    if a.command in ("language", "verify", "all"):
        verify(a.out, a.data, device)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile transfer_v2/training.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import time

import numpy as np
import torch
from torch.nn import functional as F

from .common import dump, effective_tokens, lr_at, seed_all, sha_file, sync
from .model import LM, language_model
from .nca import control_probabilities, evaluate_controls, targets


def append(path, row):
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, allow_nan=False)+"\n")


def batch(array, c, rng, device, synthetic):
    length, size = c["model"]["context"], c["batch"]
    if synthetic:
        indices = rng.integers(0, len(array), size=size)
        values = array[indices, :length+1]
    else:
        if len(array) <= length:
            raise ValueError("Language training split is shorter than context")
        indices = rng.integers(0, len(array)-length, size=size)
        values = array[indices[:, None]+np.arange(length+1)[None]]
    tensor = torch.as_tensor(values.astype(np.int64), device=device)
    labels = targets(tensor, c) if synthetic else tensor[:, 1:]
    return tensor[:, :-1], labels, indices


def eval_rows(array, c, synthetic):
    length = c["model"]["context"]
    if synthetic:
        return array[:min(len(array), c["ppt_eval_rules"]), :length+1]
    count = min(c["language_eval_tokens"]//length, (len(array)-1)//length)
    if count < 1:
        raise ValueError("Evaluation split too short")
    # Target tokens do not overlap across rows. Adjacent rows share only one
    # conditioning boundary token, as in the historical non-overlap evaluation.
    return np.stack([array[i*length:i*length+length+1] for i in range(count)])


@torch.no_grad()
def evaluate(model, array, c, device, synthetic):
    model.eval()
    rows = eval_rows(array, c, synthetic)
    total_loss = patch_loss = 0.
    count = patch_count = correct = 0
    for start in range(0, len(rows), c["eval_batch"]):
        z = torch.as_tensor(rows[start:start+c["eval_batch"]].astype(np.int64), device=device)
        labels = targets(z, c) if synthetic else z[:, 1:]
        logits = model(z[:, :-1])
        losses = F.cross_entropy(logits.transpose(1, 2), labels, reduction="none", ignore_index=-100)
        mask = labels != -100
        total_loss += float(losses[mask].sum())
        count += int(mask.sum())
        if synthetic:
            patch_mask = mask & (labels < c["nca"]["states"]**4)
            patch_loss += float(losses[patch_mask].sum())
            patch_count += int(patch_mask.sum())
            correct += int(((logits.argmax(-1) == labels) & patch_mask).sum())
    model.train()
    result = {"nll": total_loss/count, "targets": count}
    if synthetic:
        result.update(patch_nll=patch_loss/patch_count, patch_accuracy=correct/patch_count,
                      patch_targets=patch_count, rules=len(rows))
    return result


def optimizer_for(model, stage, synthetic):
    if synthetic:
        return torch.optim.Adam(model.parameters(), lr=stage["lr"], weight_decay=stage["weight_decay"])
    return torch.optim.AdamW(model.parameters(), lr=stage["lr"], weight_decay=stage["weight_decay"])


def update(model, optimizer, array, c, stage, local_step, rng, device, synthetic, order_hash):
    lr = lr_at(local_step, stage)
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.
    prediction_targets = 0
    for _ in range(c["accumulate"]):
        x, y, indices = batch(array, c, rng, device, synthetic)
        order_hash.update(indices.astype("<i8").tobytes())
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, model.vocab), y.reshape(-1), ignore_index=-100)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite loss")
        (loss/c["accumulate"]).backward()
        loss_sum += float(loss.detach())/c["accumulate"]
        prediction_targets += int((y != -100).sum())
    if stage["clip"] is not None:
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), stage["clip"], error_if_nonfinite=True)
        if not torch.isfinite(norm):
            raise FloatingPointError("Non-finite gradient norm")
    else:
        # A finite global-norm check without clipping; avoids synchronizing the
        # GPU separately for every parameter tensor.
        torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"), error_if_nonfinite=True)
    optimizer.step()
    return loss_sum, prediction_targets, lr


def quality_gate(initial, selected, controls, c):
    relative = (initial["patch_nll"]-selected["patch_nll"])/initial["patch_nll"]
    baseline = min(controls["unigram_patch_nll"], controls["copy_mixture_patch_nll"])
    margin = baseline-selected["patch_nll"]
    passed = relative >= c["gate"]["minimum_relative_improvement"] and margin >= c["gate"]["baseline_margin_nats"]
    return {"passed": bool(passed), "enforced": c["gate"]["enforce"],
            "relative_patch_nll_improvement": relative, "improvement_over_best_control_nats": margin,
            "required_relative_improvement": c["gate"]["minimum_relative_improvement"],
            "required_baseline_margin_nats": c["gate"]["baseline_margin_nats"],
            "control_scope": "Unseen-rule validation; control probabilities fitted only on synthetic train prefixes"}


def train_ppt(c, arm, seed, arrays, manifest_hash, root, device):
    if arm not in ("grid_nca", "shuffled_nca"):
        raise ValueError("Scratch has no PPT stage")
    sync(device)
    started = time.perf_counter()
    folder = Path(root)/f"{arm}_{seed}"
    folder.mkdir(parents=True, exist_ok=False)
    seed_all(seed)
    kind = "nca" if arm == "grid_nca" else "shuffled"
    train, validation = arrays[f"{kind}_train"], arrays[f"{kind}_validation"]
    model = LM(c["model"], c["nca"]["states"]**4+2, core_seed=seed, head_seed=seed+50_000).to(device)
    initial_core = model.core_hash()
    stage = c["ppt"]
    optimizer = optimizer_for(model, stage, synthetic=True)
    rng = np.random.default_rng(seed+101)
    order_hash = hashlib.sha256()
    probabilities, mixture = control_probabilities(train, c)
    selected_rows = eval_rows(validation, c, True)
    controls = evaluate_controls(selected_rows, c, probabilities, mixture)
    initial = evaluate(model, validation, c, device, True)
    best, best_step = initial, 0
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    train_seconds = 0.
    predicted = 0
    window_loss = 0.
    window_count = 0
    append(folder/"ppt_metrics.jsonl", {"ppt_step": 0, "train_loss": None, "validation": initial,
                                       "train_seconds": 0., "stage_seconds": time.perf_counter()-started})
    for k in range(stage["steps"]):
        sync(device)
        tick = time.perf_counter()
        loss, count, lr = update(model, optimizer, train, c, stage, k, rng, device, True, order_hash)
        sync(device)
        train_seconds += time.perf_counter()-tick
        window_loss += loss
        window_count += 1
        predicted += count
        if (k+1) % c["eval_every"] == 0 or k+1 == stage["steps"]:
            measured = evaluate(model, validation, c, device, True)
            sync(device)
            if measured["patch_nll"] < best["patch_nll"]:
                best, best_step = measured, k+1
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            row = {"ppt_step": k+1, "ppt_tokens": (k+1)*effective_tokens(c),
                   "train_loss": window_loss/window_count, "lr": lr, "validation": measured,
                   "train_seconds": train_seconds, "stage_seconds": time.perf_counter()-started}
            append(folder/"ppt_metrics.jsonl", row)
            print(f"PPT {arm} seed={seed} {k+1}/{stage['steps']}: held-out patch NLL={measured['patch_nll']:.4f}, best control={min(controls['unigram_patch_nll'], controls['copy_mixture_patch_nll']):.4f}", flush=True)
            window_loss = 0.
            window_count = 0
    checkpoint = {"schema_version": 2, "config": c, "stage": "ppt", "arm": arm, "seed": seed,
                  "vocab": model.vocab, "data_manifest_sha256": manifest_hash,
                  "core_initial_sha256": initial_core, "selected_step": best_step,
                  "model": best_state, "validation": best}
    torch.save(checkpoint, folder/"ppt_selected.pt")
    # Preserve both selected and final states if validation selection differs.
    if best_step != stage["steps"]:
        final_checkpoint = dict(checkpoint, model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                                selected_step=stage["steps"], validation=measured)
        torch.save(final_checkpoint, folder/"ppt_final.pt")
    gate = quality_gate(initial, best, controls, c)
    summary = {"arm": arm, "seed": seed, "ppt_updates": stage["steps"],
               "ppt_tokens": stage["steps"]*effective_tokens(c), "ppt_prediction_targets": predicted,
               "initial": initial, "selected": best, "selected_step": best_step,
               "controls": controls, "gate": gate, "core_initial_sha256": initial_core,
               "train_order_sha256": order_hash.hexdigest(), "train_seconds": train_seconds,
               "stage_seconds": time.perf_counter()-started,
               "checkpoint": "ppt_selected.pt", "checkpoint_sha256": sha_file(folder/"ppt_selected.pt"),
               "final_checkpoint_sha256": sha_file(folder/"ppt_final.pt") if (folder/"ppt_final.pt").exists() else sha_file(folder/"ppt_selected.pt"),
               "selection_policy": "Minimum held-out-rule patch NLL; full declared PPT budget is charged regardless of selected step"}
    dump(folder/"ppt_summary.json", summary)
    return summary


def load_ppt_core(folder, c, arm, seed, manifest_hash):
    folder = Path(folder)
    summary = json.loads((folder/"ppt_summary.json").read_text())
    path = folder/summary["checkpoint"]
    if sha_file(path) != summary["checkpoint_sha256"]:
        raise ValueError("PPT checkpoint checksum mismatch")
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if saved["config"] != c or saved["arm"] != arm or saved["seed"] != seed or saved["data_manifest_sha256"] != manifest_hash:
        raise ValueError("PPT checkpoint identity mismatch")
    core = {k[len("core."):]: v for k, v in saved["model"].items() if k.startswith("core.")}
    return core, summary


def train_language(c, arm, seed, arrays, manifest, manifest_hash, root, device):
    sync(device)
    setup_started = time.perf_counter()
    folder = Path(root)/f"{arm}_{seed}"
    folder.mkdir(parents=True, exist_ok=True)
    if (folder/"language_metrics.jsonl").exists():
        raise FileExistsError("Refusing to mix language runs; use a fresh output directory")
    seed_all(seed)
    ppt_state, ppt = (None, None) if arm == "scratch" else load_ppt_core(folder, c, arm, seed, manifest_hash)
    model = language_model(c, manifest["text"]["vocab"], seed, ppt_state).to(device)
    start_core, start_head = model.core_hash(), model.head_hash()
    stage = c["language"]
    optimizer = optimizer_for(model, stage, synthetic=False)
    if optimizer.state:
        raise RuntimeError("Language optimizer must start empty")
    rng = np.random.default_rng(seed+202)
    order_hash, schedule_hash = hashlib.sha256(), hashlib.sha256()
    train_seconds = 0.
    nca_prepare = sum(v["prepare_seconds"]-(v["shuffle_seconds"] if arm == "grid_nca" else 0.) for v in manifest["nca"].values()) if ppt else 0.
    setup_cost = manifest["text"]["seconds"]+nca_prepare+manifest.get("shared_finalize_seconds", 0.)
    ppt_cost = ppt["stage_seconds"] if ppt else 0.
    sync(device)
    setup_cost += time.perf_counter()-setup_started
    started = time.perf_counter()

    def record(step, train_loss, lr):
        value = evaluate(model, arrays["text_validation"], c, device, False)
        sync(device)
        elapsed = time.perf_counter()-started
        row = {"arm": arm, "seed": seed, "language_step": step,
               "language_tokens": step*effective_tokens(c),
               "ppt_tokens": ppt["ppt_tokens"] if ppt else 0,
               "total_training_tokens": step*effective_tokens(c)+(ppt["ppt_tokens"] if ppt else 0),
               "train_loss": train_loss, "lr": lr, "validation_nll": value["nll"],
               "validation_targets": value["targets"], "language_train_seconds": train_seconds,
               "language_seconds": elapsed, "cumulative_seconds": setup_cost+ppt_cost+elapsed}
        append(folder/"language_metrics.jsonl", row)
        print(f"LANGUAGE {arm} seed={seed} {step}/{stage['steps']}: validation NLL={value['nll']:.4f}", flush=True)
        return row

    first = record(0, None, None)
    window_loss = 0.
    window_count = 0
    for k in range(stage["steps"]):
        sync(device)
        tick = time.perf_counter()
        loss, _, lr = update(model, optimizer, arrays["text_train"], c, stage, k, rng, device, False, order_hash)
        sync(device)
        train_seconds += time.perf_counter()-tick
        schedule_hash.update(np.array([lr], dtype="<f8").tobytes())
        window_loss += loss
        window_count += 1
        if (k+1) % c["eval_every"] == 0 or k+1 == stage["steps"]:
            last = record(k+1, window_loss/window_count, lr)
            window_loss = 0.
            window_count = 0
    # Final validation is never selected using a test set. Confirmation data are
    # neither downloaded nor evaluated unless enabled in the pre-run config.
    test = evaluate(model, arrays["text_test"], c, device, False) if c["text"]["evaluate_test"] else None
    checkpoint = {"schema_version": 2, "config": c, "stage": "language", "arm": arm, "seed": seed,
                  "vocab": model.vocab, "data_manifest_sha256": manifest_hash,
                  "model": model.state_dict(), "language_step": stage["steps"],
                  "validation": {"nll": last["validation_nll"], "targets": last["validation_targets"]},
                  "test": test}
    torch.save(checkpoint, folder/"language_final.pt")
    summary = {"arm": arm, "seed": seed, "purpose": c["purpose"],
               "language_updates": stage["steps"], "language_tokens": stage["steps"]*effective_tokens(c),
               "ppt_updates": c["ppt"]["steps"] if ppt else 0, "ppt_tokens": ppt["ppt_tokens"] if ppt else 0,
               "parameters": sum(p.numel() for p in model.parameters()),
               "core_initial_sha256": ppt["core_initial_sha256"] if ppt else start_core,
               "language_start_core_sha256": start_core, "language_head_initial_sha256": start_head,
               "language_optimizer_initial_state_entries": 0,
               "language_train_order_sha256": order_hash.hexdigest(), "language_schedule_sha256": schedule_hash.hexdigest(),
               "initial_validation_nll": first["validation_nll"], "final_validation_nll": last["validation_nll"],
               "language_train_seconds": train_seconds, "language_seconds": last["language_seconds"],
               "cumulative_seconds": last["cumulative_seconds"], "test": test,
               "checkpoint": "language_final.pt", "checkpoint_sha256": sha_file(folder/"language_final.pt")}
    dump(folder/"language_summary.json", summary)
    return summary


In [ ]:
%%writefile transfer_v2/verify.py
from __future__ import annotations

import json
from pathlib import Path

import torch

from .common import ARMS, dump, effective_tokens, sha_file
from .data import load_data
from .model import LM
from .training import evaluate


def verify(root, data_root, device="cpu"):
    root = Path(root)
    env = json.loads((root/"environment.json").read_text())
    c = env["config"]
    arrays, _ = load_data(data_root, c)
    manifest_hash = sha_file(Path(data_root)/"manifest.json")
    if manifest_hash != env["data_manifest_sha256"]:
        raise ValueError("Data manifest changed")
    records, language = [], {}
    selected_count = 0
    extra_final_count = 0
    for arm in ARMS:
        for seed in c["seeds"]:
            folder = root/f"{arm}_{seed}"
            for stage in ("ppt", "language"):
                path = folder/f"{stage}_summary.json"
                if not path.exists():
                    continue
                summary = json.loads(path.read_text())
                checkpoint = folder/summary["checkpoint"]
                if sha_file(checkpoint) != summary["checkpoint_sha256"]:
                    raise ValueError(f"Checkpoint changed: {checkpoint.name}")
                saved = torch.load(checkpoint, map_location=device, weights_only=True)
                if saved["config"] != c or saved["arm"] != arm or saved["seed"] != seed or saved["stage"] != stage:
                    raise ValueError("Checkpoint metadata mismatch")
                if saved["data_manifest_sha256"] != manifest_hash:
                    raise ValueError("Checkpoint/data mismatch")
                model = LM(c["model"], saved["vocab"], core_seed=seed, head_seed=seed)
                model.load_state_dict(saved["model"], strict=True)
                model.to(device)
                if stage == "ppt":
                    key = "nca_validation" if arm == "grid_nca" else "shuffled_validation"
                    measured = evaluate(model, arrays[key], c, device, True)
                    errors = {k: abs(measured[k]-saved["validation"][k]) for k in ("nll", "patch_nll", "patch_accuracy")}
                else:
                    if summary["language_updates"] != c["language"]["steps"] or summary["language_tokens"] != c["language"]["steps"]*effective_tokens(c):
                        raise ValueError("Language budgets differ")
                    measured = evaluate(model, arrays["text_validation"], c, device, False)
                    errors = {"validation_nll": abs(measured["nll"]-saved["validation"]["nll"])}
                    if saved["test"] is not None:
                        test = evaluate(model, arrays["text_test"], c, device, False)
                        errors["test_nll"] = abs(test["nll"]-saved["test"]["nll"])
                    language[arm, seed] = summary
                if any(v > 1e-4 for v in errors.values()):
                    raise ValueError(f"Reloaded loss differs: {arm}, {seed}, {stage}, {errors}")
                records.append({"arm": arm, "seed": seed, "stage": stage,
                                "checksum_valid": True, "absolute_errors": errors})
                selected_count += 1
                print(f"Verified {stage} {arm} seed={seed}: max error {max(errors.values()):.3g}", flush=True)
                if stage == "ppt" and (folder/"ppt_final.pt").exists():
                    if sha_file(folder/"ppt_final.pt") != summary["final_checkpoint_sha256"]:
                        raise ValueError("Final PPT checkpoint checksum mismatch")
                    final = torch.load(folder/"ppt_final.pt", map_location=device, weights_only=True)
                    if final["config"] != c or final["arm"] != arm or final["seed"] != seed or final["data_manifest_sha256"] != manifest_hash:
                        raise ValueError("Final PPT checkpoint identity mismatch")
                    model.load_state_dict(final["model"], strict=True)
                    measured = evaluate(model, arrays[key], c, device, True)
                    final_errors = {k: abs(measured[k]-final["validation"][k]) for k in ("nll", "patch_nll", "patch_accuracy")}
                    if max(final_errors.values()) > 1e-4:
                        raise ValueError("Final PPT checkpoint loss mismatch")
                    records.append({"arm": arm, "seed": seed, "stage": "ppt_final",
                                    "absolute_errors": final_errors, "sha256": sha_file(folder/"ppt_final.pt")})
                    extra_final_count += 1
    for seed in c["seeds"]:
        summaries = [language[a, seed] for a in ARMS if (a, seed) in language]
        if summaries and len(summaries) != len(ARMS):
            raise ValueError("Incomplete language arm set")
        for key in ("core_initial_sha256", "language_head_initial_sha256", "language_train_order_sha256",
                    "language_schedule_sha256", "language_tokens", "language_updates"):
            if len({s[key] for s in summaries}) > 1:
                raise ValueError(f"Unmatched language control: {key}, seed {seed}")
        if any(s["language_optimizer_initial_state_entries"] != 0 for s in summaries):
            raise ValueError("Language optimizer was not fresh")
    status = json.loads((root/"status.json").read_text())
    expected = len(c["seeds"])*(5 if status["state"] == "complete" else 2)
    if selected_count != expected:
        raise ValueError(f"Expected {expected} selected-stage checkpoints; verified {selected_count}")
    result = {"device": device, "status": status, "stage_checkpoints_verified": len(records),
              "selected_stage_checkpoints_verified": selected_count,
              "additional_final_ppt_checkpoints_verified": extra_final_count,
              "records": records, "language_controls_checked": bool(language),
              "note": "Checkpoint consistency does not prove learning acceleration"}
    dump(root/"verification.json", result)
    return result


In [ ]:
%%writefile configs/transfer_v2.json
{
  "schema_version": 2,
  "purpose": "development",
  "seeds": [17, 29, 43],
  "precision": "fp32",
  "model": {"layers": 6, "heads": 4, "width": 192, "context": 1024},
  "batch": 2,
  "accumulate": 2,
  "eval_every": 100,
  "eval_batch": 2,
  "language_eval_tokens": 32768,
  "ppt_eval_rules": 32,
  "ppt": {"steps": 1000, "lr": 0.0001, "warmup_fraction": 0.1, "weight_decay": 0.0, "clip": null},
  "language": {"steps": 2000, "lr": 0.0005, "warmup_fraction": 0.1, "weight_decay": 0.0001, "clip": 1.0},
  "nca": {
    "grid": 12, "states": 10, "burnin": 10, "temperature": 0.0001,
    "profile_frames": 10, "filter_low": 0.5, "filter_high": 1.0,
    "train_rules": 4096, "val_rules": 128, "candidate_batch": 64,
    "max_candidates": 262144, "data_seed": 20261018,
    "channel_padding": "spatial_only"
  },
  "gate": {"enforce": true, "minimum_relative_improvement": 0.05, "baseline_margin_nats": 0.01},
  "text": {
    "dataset": "Salesforce/wikitext", "subset": "wikitext-2-raw-v1",
    "revision": "b08601e04326c79dfdd32d625aee71d232d685c3",
    "tokenizer": "gpt2", "evaluate_test": false
  }
}


In [ ]:
%%writefile configs/transfer_v2_smoke.json
{
  "schema_version": 2,
  "purpose": "execution_check",
  "seeds": [17],
  "precision": "fp32",
  "model": {"layers": 2, "heads": 2, "width": 32, "context": 76},
  "batch": 2,
  "accumulate": 1,
  "eval_every": 2,
  "eval_batch": 2,
  "language_eval_tokens": 304,
  "ppt_eval_rules": 4,
  "ppt": {"steps": 4, "lr": 0.001, "warmup_fraction": 0.25, "weight_decay": 0.0, "clip": null},
  "language": {"steps": 6, "lr": 0.001, "warmup_fraction": 0.2, "weight_decay": 0.0001, "clip": 1.0},
  "nca": {
    "grid": 12, "states": 10, "burnin": 10, "temperature": 0.0,
    "profile_frames": 10, "filter_low": 0.0, "filter_high": 1.1,
    "train_rules": 8, "val_rules": 4, "candidate_batch": 8,
    "max_candidates": 128, "data_seed": 20261018,
    "channel_padding": "spatial_only"
  },
  "gate": {"enforce": false, "minimum_relative_improvement": 0.05, "baseline_margin_nats": 0.01},
  "text": {"dataset": "offline_fixture", "subset": null, "revision": null, "tokenizer": "byte_fixture", "evaluate_test": false}
}


In [ ]:
%%writefile tests/test_transfer_v2.py
"""Regression coverage for independent-stage scientific controls."""
import copy
from pathlib import Path
import unittest

import numpy as np
import torch
from torch.nn import functional as F

from transfer_v2.common import effective_tokens, load_config, validate_config
from transfer_v2.model import LM, language_model
from transfer_v2.nca import (complexity, convolution_logits, encode_frames, generate_split, lattice,
                             message_logits, sample_rules, shuffled, targets)
from transfer_v2.report import first_hit
from transfer_v2.training import optimizer_for, quality_gate


ROOT = Path(__file__).resolve().parents[1]


class TransferV2Controls(unittest.TestCase):
    def setUp(self):
        torch.set_num_threads(2)
        self.c = load_config(ROOT/"configs/transfer_v2_smoke.json")

    def test_real_configuration_has_independent_budgets(self):
        c = load_config(ROOT/"configs/transfer_v2.json")
        self.assertEqual(c["ppt"]["steps"], 1000)
        self.assertEqual(c["language"]["steps"], 2000)
        self.assertEqual(effective_tokens(c), 4096)
        self.assertEqual(c["nca"]["states"], 10)
        self.assertTrue(c["gate"]["enforce"])
        self.assertFalse(c["text"]["evaluate_test"])

    def test_default_model_accepts_real_bpe_vocabulary_and_backpropagates(self):
        c = load_config(ROOT/"configs/transfer_v2.json")
        model = language_model(c, 50257, 17)
        logits = model(torch.zeros((1, 16), dtype=torch.long))
        self.assertEqual(tuple(logits.shape), (1, 16, 50257))
        F.cross_entropy(logits.reshape(-1, 50257), torch.zeros(16, dtype=torch.long)).backward()
        self.assertTrue(torch.isfinite(model.core.blocks[0].qkv.weight.grad).all())
        self.assertGreater(sum(p.numel() for p in model.parameters()), 10_000_000)

    def test_development_cannot_bypass_synthetic_gate(self):
        c = load_config(ROOT/"configs/transfer_v2.json")
        c["gate"]["enforce"] = False
        with self.assertRaises(ValueError):
            validate_config(c)

    def test_message_passing_matches_categorical_spatial_convolution(self):
        g = torch.Generator().manual_seed(17)
        rules = sample_rules(3, 10, g, "cpu")
        x = torch.randint(10, (3, 12*12), generator=g)
        a = message_logits(x, torch.tensor(lattice(12)), rules, 10)
        b = convolution_logits(x, rules, 12, 10)
        torch.testing.assert_close(a, b, atol=1e-6, rtol=1e-5)
        torch.testing.assert_close(a.argmax(-1), b.argmax(-1))

    def test_categorical_patch_encoding_is_bijective(self):
        ids = torch.arange(10000)
        cells = torch.stack([(ids//10**j) % 10 for j in range(4)], -1)[:, None]
        encoded = encode_frames(cells, 2, 10)
        torch.testing.assert_close(encoded[:, 1], ids)
        self.assertTrue(torch.all(encoded[:, 0] == 10000))
        self.assertTrue(torch.all(encoded[:, 2] == 10001))

    def test_initial_context_mask_and_patch_only_metric(self):
        frames = torch.randint(10, (2, 3, 144))
        sequence = encode_frames(frames, 12, 10)[:, :77]
        y = targets(sequence, self.c)
        self.assertTrue(torch.all(y[:, :37] == -100))
        self.assertTrue(torch.all(y[:, 37:] >= 0))
        patches = targets(sequence, self.c, patch_only=True)
        self.assertEqual(int((patches != -100).sum()), 72)
        self.assertTrue(torch.all(patches[patches != -100] < 10000))

    def test_gzip_score_ignores_markers_and_integer_input_width(self):
        x = encode_frames(torch.randint(10, (2, 10, 144)), 12, 10).numpy()
        changed = x.copy().reshape(2, 10, 38)
        changed[:, :, 0] = 32000
        changed[:, :, -1] = 32001
        np.testing.assert_array_equal(complexity(x.astype(np.uint16), self.c),
                                      complexity(changed.reshape(2, -1).astype(np.int64), self.c))

    def test_shuffled_control_preserves_counts_and_marker_locations(self):
        x = encode_frames(torch.randint(10, (8, 3, 144)), 12, 10).numpy().astype(np.uint16)
        y = shuffled(x, self.c, 123)
        np.testing.assert_array_equal(np.sort(x, axis=1), np.sort(y, axis=1))
        np.testing.assert_array_equal(x >= 10000, y >= 10000)
        self.assertFalse(np.array_equal(x, y))

    def test_rule_split_generation_is_reproducible_and_disjoint(self):
        a, ra, sa = generate_split(self.c, 3, "train", "cpu")
        b, rb, sb = generate_split(self.c, 3, "train", "cpu")
        _, rv, sv = generate_split(self.c, 3, "validation", "cpu")
        np.testing.assert_array_equal(a, b)
        np.testing.assert_array_equal(ra["perception"], rb["perception"])
        self.assertEqual(sa["rule_ids"], sb["rule_ids"])
        self.assertFalse(set(sa["rule_ids"]) & set(sv["rule_ids"]))
        self.assertFalse(np.array_equal(ra["perception"], rv["perception"]))

    def test_filter_never_silently_relaxes_threshold(self):
        c = copy.deepcopy(self.c)
        c["nca"].update(filter_low=2., filter_high=3., max_candidates=8)
        with self.assertRaisesRegex(RuntimeError, "no silent threshold relaxation"):
            generate_split(c, 1, "train", "cpu")

    def test_backbone_pairing_is_independent_of_vocabulary(self):
        a = LM(self.c["model"], 10002, 17, 50)
        b = LM(self.c["model"], 256, 17, 123)
        self.assertEqual(a.core_hash(), b.core_hash())

    def test_transfer_copies_backbone_and_reinitializes_common_head(self):
        ppt = LM(self.c["model"], 10002, 17, 50)
        with torch.no_grad():
            next(ppt.core.parameters()).add_(.1)
        base = language_model(self.c, 256, 17)
        transferred = language_model(self.c, 256, 17, ppt.core.state_dict())
        self.assertEqual(base.head_hash(), transferred.head_hash())
        self.assertNotEqual(base.core_hash(), transferred.core_hash())
        self.assertEqual(transferred.core_hash(), ppt.core_hash())
        self.assertTrue(all(p.requires_grad for p in transferred.parameters()))

    def test_language_optimizer_is_fresh_after_ppt(self):
        ppt = LM(self.c["model"], 10002, 17, 50)
        opt = optimizer_for(ppt, self.c["ppt"], True)
        logits = ppt(torch.zeros((2, 8), dtype=torch.long))
        F.cross_entropy(logits.reshape(-1, 10002), torch.zeros(16, dtype=torch.long)).backward()
        opt.step()
        self.assertTrue(opt.state)
        transferred = language_model(self.c, 256, 17, ppt.core.state_dict())
        language_opt = optimizer_for(transferred, self.c["language"], False)
        self.assertEqual(len(language_opt.state), 0)

    def test_causal_rope_model_cannot_see_future(self):
        model = language_model(self.c, 256, 17).eval()
        x = torch.randint(256, (2, 16))
        y = x.clone()
        y[:, 8:] = torch.randint(256, (2, 8))
        torch.testing.assert_close(model(x)[:, :8], model(y)[:, :8], rtol=0, atol=1e-6)

    def test_gate_requires_prediction_beyond_static_controls(self):
        controls = {"unigram_patch_nll": 7., "copy_mixture_patch_nll": 6.5}
        self.assertFalse(quality_gate({"patch_nll": 9.}, {"patch_nll": 7.}, controls, self.c)["passed"])
        self.assertTrue(quality_gate({"patch_nll": 9.}, {"patch_nll": 6.}, controls, self.c)["passed"])

    def test_unreached_language_reference_is_censored(self):
        curve = [{"validation_nll": 8., "language_step": 0}, {"validation_nll": 7., "language_step": 100}]
        self.assertIsNone(first_hit(curve, 6.))
        self.assertEqual(first_hit(curve, 7.)["language_step"], 100)


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile requirements.txt
numpy>=1.26,<3
torch>=2.5,<3
matplotlib>=3.8,<4


In [ ]:
%%writefile requirements-v2.txt
-r requirements.txt
tiktoken>=0.7,<1
pyarrow>=17,<27


In [ ]:
%%writefile docs/TRANSFER_V2_PROTOCOL.md
# Version 2: acceleration through NCA pre-pre-training

The primary question is whether grid-NCA pre-pre-training produces a transformer
initialization that learns natural language **faster during a subsequent,
otherwise identical language-training stage**. A synthetic-stage diagnostic is a
prerequisite; it is not itself evidence of language transfer.

This implementation is separate from historical `experiment.py`. Pilot 1 and its
artifacts retain their original identities and limitations.

## Default stages and comparisons

Configuration: `configs/transfer_v2.json`.

| Arm | Synthetic pre-pre-training | Language pre-training |
| --- | --- | --- |
| `scratch` | None | 2000 identical language updates |
| `grid_nca` | 1000 grid-NCA updates | 2000 identical language updates |
| `shuffled_nca` | 1000 shuffled-grid updates | 2000 identical language updates |

Three paired model seeds: 17, 29, 43. The graph-topology extension is deliberately
deferred until the grid reference shows useful transfer. The grid update is
already expressed as relational message passing, with an independently tested
equivalent spatial convolution. Rewiring is not silently mixed into this test.

Effective batch: 2 sequences per microbatch, 2 gradient-accumulation microbatches,
context 1024. Each update consumes 4096 input tokens. Each synthetic arm/seed gets
4,096,000 synthetic tokens and every arm/seed gets **8,192,000 language tokens**.
There are six PPT stages and nine language stages: 98,304,000 model-input tokens
in total if the gate permits all language runs. Masked conditioning tokens are
included in exposure counts; supervised synthetic-target counts are also logged.

All language arms start at **language step zero**, with identical:

- Initial language embedding/output weights (paired by seed, explicitly tied).
- Language data-window order, effective batch, context, validation subset and N.
- Fresh AdamW state, LR schedule, weight decay and clipping settings.
- Backbone architecture and original pre-PPT backbone initialization per seed.

The two PPT arms transfer only their selected backbone state. Synthetic symbol
embeddings, Adam moments and scheduler state are not transferred. Scratch begins
with the paired untrained backbone and receives no preceding target-text updates.
Checksums of language heads, batch orders and schedules are checked across arms.

## Model and stage-specific optimization

The model uses six layers, width 192, four attention heads, RMSNorm, rotary
position encoding and SwiGLU, without dropout. It is a small Llama-style backbone,
not the paper's full-sized model or a bit-exact Hugging Face reproduction.
Backbone initialization is isolated from embedding initialization so changing
vocabulary size cannot alter the paired original backbone.

The synthetic vocabulary is 10,002: 10,000 categorical patch IDs and two markers.
The language vocabulary is GPT-2's 50,257 BPE IDs. Parameter counts differ across
stages because embeddings differ: **4,577,088 during PPT** and **12,306,048 during
language training**. Language arms are identical in size. Counts are also
recorded by the on-device calibration and run summaries.

- PPT: Adam, LR 1e-4, no weight decay, no gradient clipping. A finite global-norm
  check still stops non-finite training.
- Language: AdamW, LR 5e-4, weight decay 1e-4, gradient clipping at 1.0.
- Both: independent 10% warmup and cosine decay to 10% of their stage LR.
- Precision: fp32, deterministic PyTorch algorithms, TF32 disabled. This is an
  auditability choice, not a maximum-throughput GPU-kernel comparison.

## Categorical NCA generation

Based on the [author's generator and trainer](https://github.com/danihyunlee/nca-pre-pretraining/tree/bdd1c71e04eb2e5a2c91481faf278e95846b4431),
retained with its MIT license in the source reference directory:

- Periodic 12x12 grid with 10 categorical states, represented one-hot.
- Shared 3x3 perception with four channels, then a 16-channel pointwise layer,
  ReLU and ten next-state logits. Biases are zero; kernels use a fan-in-normal
  distribution with truncated-normal variance correction.
- Frozen randomly sampled neural rule per trajectory, low-temperature categorical
  sampling (1e-4, matching the pinned launcher).
- Ten burn-in steps before recording. Profile ten frames for rule selection.
- Each 2x2 patch maps bijectively to a base-10 integer. Frame markers bracket the
  patch sequence. A 1024-token context spans about 27 grid frames.
- Rule-profile filtering uses **0.5 < gzip ratio < 1.0**, with marker-free
  little-endian int32 patch IDs, gzip level 9. Bounds are not silently relaxed.
- Actual cached trajectories use fresh initial conditions after screening;
  their measured compression ratios can differ from the profile band.
- 4096 retained training rules and 128 retained held-out rules, using separate
  generation streams. Model validation uses the first 32 held-out rules, fixed
  before training. Rule identities, frozen rule weights, token files, scores,
  acceptance rates, collapse rates, configuration and hashes are retained.

The cache is reused across model seeds, so this is three model seeds on one
synthetic dataset, not independent data/topology replication. Synthetic sampling
uses replacement from the cached trajectories. The shuffle control preserves
each trajectory's full patch histogram and exact marker positions, while
destroying patch order. Because context is a prefix, its prefix histogram need
not equal the unshuffled prefix histogram; native training-prefix controls are
fitted separately. A more tightly prefix-matched shuffle is a future ablation.

Explicit choices relative to reference quirks: spatial axes alone are padded;
initial categorical states are uniform as described in the paper; no accidental
channel padding is copied. This port is tested algebraically, but not claimed
bit-identical to JAX/Flax random initialization or the reference's PRNG streams.
See `METHODOLOGY_AUDIT.md` for the paper/code discrepancies.

## Verify synthetic learning before language training

Training examples are trajectory-aligned. The first complete grid is conditioning
context, excluded from the loss. Frame-marker prediction remains in the training
objective, matching the effective reference dataset behavior. Evaluation also
reports **patch-only** NLL and patch accuracy so marker prediction cannot satisfy
the quality check.

At step zero and every 100 updates, record synthetic train loss, held-out-rule
validation loss, patch-only NLL/accuracy and timing. Select the PPT checkpoint with
minimum held-out-rule patch NLL. Save it before transfer, and retain the final
PPT state too when it differs. The entire declared PPT training cost is charged,
even if the selected checkpoint is earlier.

Controls are fitted on synthetic training prefixes only:

- Smoothed patch unigram probabilities.
- A mixture of those probabilities and copying the patch at the same position
  in the previous grid. The mixture weight is selected on training data from a
  fixed 100-point grid, not on held-out rules.

For every grid-PPT model seed, the operational admission gate requires:

1. At least **5% relative reduction** in held-out-rule patch NLL from initialization.
2. Patch NLL at least **0.01 nats below both fitted controls**.

If any grid seed fails, the default workflow saves `ppt_gate.json`, selected
checkpoints, diagnostics and verification, then finishes with status
`stopped_at_synthetic_gate`. **No language models are trained.** This means the
configuration failed the predeclared prerequisite; it does not refute NCA
transfer in general. Do not silently relax the gate to obtain language results.
It is an engineering prerequisite added here, not a claim that the paper used
this exact admission rule. Passing it does not establish useful language transfer.

Only explicitly labeled execution-check fixtures can bypass the gate. They
cannot be used as scientific acceleration evidence.

## Fresh language data and reserved confirmation

The default language data are **WikiText-2 raw**, official train/validation
splits, from the [Salesforce dataset](https://huggingface.co/datasets/Salesforce/wikitext)
at revision `b08601e04326c79dfdd32d625aee71d232d685c3`. They do not reuse the
Shakespeare test set inspected in pilot 1. Parquet files are downloaded from
pinned URLs, joined with newline separators, encoded with GPT-2 BPE and hashed.
The tokenizer's actual merge table and special-token mapping are fingerprinted.
These downloads require Internet in the Kaggle notebook.

Language training draws random windows with replacement. It is an affordable
development corpus, not the paper's full OpenWebText corpus. Evaluation uses
32,768 fixed, non-overlapping target tokens from validation, with full context
blocks and no cross-split concatenation. Every arm sees identical sampled
training-window starts within a seed; hashes verify the match.

By default `evaluate_test=false`: the official confirmation test split is
**neither downloaded nor evaluated**. Do development decisions on validation
results. To schedule a confirmation evaluation, create a new predeclared config
with `evaluate_test=true` before preparation; then the final model of each arm is
evaluated once on its fixed test subset. Do not switch the flag after examining
test results or reuse an incompatible data directory.

WikiText content has its own CC-BY-SA/GFDL terms. Downloaded corpora and weights
are local run artifacts, excluded from publication inventories.

## Measure learning acceleration

Primary plots and metrics use **language-stage tokens/time starting at zero**.
For each seed, the predeclared reference target is the scratch arm's final
validation loss at N language updates. The method is fixed before running; the
numerical target is derived solely from that paired reference validation run.

For each synthetic arm, locate the first recorded evaluation reaching that
target, without interpolation. Report language-token and language-time speedups
as scratch cost divided by the synthetic-initialized arm's language cost.
Unreached targets remain censored (`null`). Never average only successful seeds:
an all-seed aggregate speedup is withheld if any paired seed misses the target.

Separately report total input tokens and measured cumulative cost, adding PPT
and relevant preparation to language training. A language-stage speedup can be
real while failing to repay PPT cost; report both findings explicitly.

Language seconds include training and fixed validation, starting after the
language model/head/optimizer are constructed. Cumulative seconds also charge
that setup, text preparation, relevant synthetic generation/cache writes,
shared manifest work and PPT stage costs. Synthetic preparation is conservatively
charged in full per arm/seed although the suite reuses its cache. They exclude
optional disposable calibration, package/runtime startup, cloud queueing and
work after the last validation record such as final checkpoint save, verification,
plots and ZIP creation. Neither metric is exact cloud-billing wall time.

The notebook calibrates actual update/evaluation timing on disposable models
first, then prints an estimate for the configured suite. That estimate excludes
several costs and is not a runtime guarantee. Calibration weights never transfer
into any research model.

## Running and completion

Import **`kaggle_nca_transfer_v2.ipynb`** into a private Kaggle notebook. Enable an
available GPU and Internet, then run all cells. It embeds the exact source and
configuration, executes validity checks and a tiny GPU execution fixture, prepares
real data, calibrates, runs PPT diagnostics and conditionally runs the matched
language benchmark. It uses one GPU.

The last cell exports **`nca_transfer_v2_results_<timestamp>.zip`** and explicitly
prints either `complete` or `stopped_at_synthetic_gate`. Both are completed
workflows; the second contains synthetic diagnostics rather than language
acceleration measurements. A Python exception is a failed run, not either status.
Save a Kaggle notebook version with outputs and download the ZIP.

CLI examples:

```bash
python -m transfer_v2.run all --config configs/transfer_v2.json --data data/transfer2 --out runs/transfer2 --device cuda
```

Separate `prepare`, `calibrate`, `ppt`, `language`, `report`, and `verify` commands
are available. The language command requires a passed PPT gate and unchanged
source/config/data identity. Never reuse existing run directories or silently
mix interrupted runs. Step-level resume is not implemented; retained PPT outputs
can feed a separate `language` command if that phase has not started.

CPU validity checks (not transfer measurements):

```bash
python scripts/run_logged.py --label v2-tests -- python -m unittest discover -s tests -p test_transfer_v2.py -v
python scripts/run_logged.py --label v2-offline -- python scripts/smoke_transfer_v2.py
```

The offline workflow checks both full completion and a deliberately failed
synthetic gate, reloading stage checkpoints and checking the language controls.
It uses generated byte fixtures, no WikiText download and no reserved test data.

New code is locally tested on CPU. GPU performance, gate outcomes and language
acceleration must be measured by the new Kaggle run; no positive result is baked
into the implementation.


In [ ]:
%%writefile LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.


In [ ]:
%%writefile NOTICE
GNNs in NCA — graph cellular automata pre-pre-training research
Copyright 2026 GNNs in NCA contributors

Original project code and documentation are licensed under Apache License 2.0.
See LICENSE. This notice does not modify any license terms.

The pinned source under sources/nca-pre-pretraining-reference/ is
Copyright (c) 2026 Dan Lee and remains under its included MIT license.
It is reference material, not a runtime dependency or an assertion of endorsement.

Tiny Shakespeare is downloaded or supplied at runtime, not included here.
Dependencies are distributed separately under their respective licenses.


In [ ]:
%%writefile THIRD_PARTY_NOTICES.md
# Third-party material

The unchanged snapshot in `sources/nca-pre-pretraining-reference/` comes from
[danihyunlee/nca-pre-pretraining](https://github.com/danihyunlee/nca-pre-pretraining),
revision `bdd1c71e04eb2e5a2c91481faf278e95846b4431` (12 March 2026).
Copyright (c) 2026 Dan Lee. It remains under the included
[MIT license](sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/LICENSE).
`reference.json` records URLs/hashes. It is not imported by the pilot; its extra
dependencies are not required for this project's checks.

The methodology audit discusses
[Training Language Models via Neural Cellular Automata, v1](https://arxiv.org/html/2603.10055v1).
The paper and competition rules PDF are not redistributed here.

The default corpus is fetched from the
[Tiny Shakespeare source](https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt)
at runtime. A data manifest records its SHA-256 and byte splits. Review the source's
terms before redistributing derived data bundles. Raw corpus bytes, cached arrays
and model checkpoints are excluded from this source snapshot.

NumPy, PyTorch and Matplotlib are installed separately under their own licenses.
Version 2 additionally uses separately installed tiktoken and PyArrow. Its NCA
port is based on the retained MIT reference, with the explicit changes recorded
in `docs/TRANSFER_V2_PROTOCOL.md`; the standalone notebook includes that MIT
license. WikiText-2 is downloaded at runtime from the pinned Salesforce dataset
revision and retains its CC-BY-SA-3.0/GFDL terms. Dataset/checkpoint artifacts
remain excluded from the public source selection.
`docs/results/` contains project measurements and plots; its provenance manifest
distinguishes copied originals from adapted summaries.


In [ ]:
%%writefile sources/nca-pre-pretraining-reference/bdd1c71e04eb2e5a2c91481faf278e95846b4431/LICENSE
MIT License

Copyright (c) 2026 Dan Lee

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


## Dependencies and controls
Kaggle supplies its CUDA PyTorch build; this cell retains it and installs only
the small tokenizer/Parquet reader if missing. A GPU is required for the actual
development run. CPU tests are engineering checks, not acceleration evidence.


In [ ]:
import importlib.metadata
from packaging.specifiers import SpecifierSet
missing = []
for package, bounds in [('tiktoken', '>=0.7,<1'), ('pyarrow', '>=17,<27')]:
    try:
        suitable = importlib.metadata.version(package) in SpecifierSet(bounds)
    except importlib.metadata.PackageNotFoundError:
        suitable = False
    if not suitable:
        missing.append(package + bounds)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator and restart the notebook runtime.'
print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))

def run(*arguments):
    subprocess.run([sys.executable, *arguments], check=True)

run('-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_transfer_v2.py', '-v')


## Tiny GPU execution fixture
This checks CUDA execution, stage checkpoints, common language controls and
verification. It deliberately uses a tiny byte fixture and does not supply a
scientific result or count toward any development model's training.


In [ ]:
fixture = Path('fixture')
fixture.mkdir()
(fixture / 'train.txt').write_text('Offline execution fixture; no research inference.\n' * 512)
(fixture / 'validation.txt').write_text('Distinct validation fixture for checks.\n' * 128)
run('-m', 'transfer_v2.run', 'all', '--config', 'configs/transfer_v2_smoke.json',
    '--data', 'data/smoke', '--out', 'runs/smoke', '--fixture-dir', str(fixture), '--device', 'cuda')


## Real development run
Data, budgets, selection and gate settings are fixed in `configs/transfer_v2.json`.
The CLI prepares filtered rule-disjoint NCA data and pinned WikiText/GPT-2 tokens,
prints an estimate measured on disposable models on your actual GPU, runs PPT,
then conditionally runs the matched language stage and verifies checkpoints.

The synthetic gate requires at least 5% held-out patch-loss improvement and a
0.01-nat advantage over both fitted unigram/copy controls for every grid model seed.
It is never relaxed automatically. Test data are neither downloaded nor evaluated.


In [ ]:
run('-m', 'transfer_v2.run', 'all', '--config', 'configs/transfer_v2.json',
    '--data', 'data/development', '--out', 'runs/development', '--device', 'cuda')


## Inspect the outcome
If status is `complete`, report language-stage learning acceleration first and
cumulative PPT+language costs separately. The target is each paired scratch
model's final validation loss. Unreached targets remain censored; all-seed
speedups are withheld when a seed misses the target. Three seeds are exploratory.

If status is `stopped_at_synthetic_gate`, this run contains diagnostics and PPT
checkpoints; it has not measured downstream learning acceleration.


In [ ]:
import json
from IPython.display import SVG, display

OUT = Path('runs/development')
status = json.loads((OUT / 'status.json').read_text())
print('RUN STATUS:', status['state'])
display(json.loads((OUT / 'ppt_gate.json').read_text()))
result = json.loads((OUT / 'report.json').read_text())
display(result['aggregate'])
display(json.loads((OUT / 'verification.json').read_text()))
display(SVG(filename=str(OUT / 'ppt_diagnostics.svg')))
if (OUT / 'learning_curves.svg').exists():
    display(SVG(filename=str(OUT / 'learning_curves.svg')))


## Download the verifiable result
Save a Kaggle notebook version with outputs, and download this ZIP. A stopped
synthetic gate is a completed diagnostic workflow, not an error or a successful
transfer claim. It is still useful to inspect and share that output archive.


In [ ]:
import zipfile
from IPython.display import FileLink

archive = Path('/kaggle/working') / f'nca_transfer_v2_results_{RUN_ID}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for path in ROOT.rglob('*'):
        if path.is_file() and not {'__pycache__', 'plot_cache'}.intersection(path.relative_to(ROOT).parts):
            z.write(path, path.relative_to(ROOT))
print('WORKFLOW FINISHED:', status['state'])
print('Download:', archive, f'({archive.stat().st_size / 1e6:.1f} MB)')
display(FileLink(str(archive)))
